# Finding models in WebLLM

This notebook is about the **model-browsing** part of WebLLM. You will see what models are available, learn how to read their names, look up their model cards, download one, check the browser cache, and delete one when you need space.

Everything runs in your browser. Nothing is sent to a server.

**Before you start:** use a recent Chrome or Edge browser on a laptop. The first download can take a while, but after that the model is saved in your browser.

## Step 1: Get set up

This checks that your browser can run models with WebGPU, loads WebLLM, and defines a few helper functions for the later steps.

In [ ]:
import js
from pyodide.ffi import to_js, create_proxy
from IPython.display import display

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")


def js_obj(d):
    """Turn a Python dict into a JavaScript object."""
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    """Show a download progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def download_model(model_id):
    """Download a model into the browser cache, then unload it from memory."""
    engine = None
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        nonlocal engine
        if chat_options:
            engine = await webllm.CreateMLCEngine(
                model_id,
                js_obj({"initProgressCallback": progress}),
                js_obj(chat_options),
            )
        else:
            engine = await webllm.CreateMLCEngine(
                model_id,
                js_obj({"initProgressCallback": progress}),
            )

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise
    finally:
        if engine is not None:
            await engine.unload()


## Step 2: See which models are available

WebLLM has a long list of models built in. The next two cells first gather the full list, then filter it.

At the top of the filter cell, you can change these settings:

- **`NEWER_THAN`**: only show models from this month or later, written as `"YYYY-MM"`. Use `""` to show all dates.
- **`MAX_SIZE_MB`**: only show models that need this much graphics memory or less.
- **`MODEL_TYPE`**: choose `"chat"`, `"embedding"`, or `"vision"`.
- **`ONLY_Q4F16`**: keep only the `q4f16` compressed versions. These are usually the best choice for students on laptops.

**About release dates:** WebLLM does not include them, so we keep a small hand-made table called `RELEASED`. If a new model family shows up with a `?`, you can add it.

In [ ]:
RELEASED = [
    ("Qwen3.5-", "2026-03"),
    ("Ministral-3-", "2025-12"),
    ("Qwen3-", "2025-04"),
    ("OLMo-2-0425", "2025-04"),
    ("gemma3-", "2025-03"),
    ("Phi-4-mini", "2025-02"),
    ("DeepSeek-R1-Distill", "2025-01"),
    ("SmolLM2-", "2024-11"),
    ("OLMo-2-1124", "2024-11"),
    ("Qwen2.5-Coder", "2024-11"),
    ("Qwen2.5-", "2024-09"),
    ("Llama-3.2-", "2024-09"),
    ("Phi-3.5", "2024-08"),
    ("Hermes-3", "2024-08"),
    ("gemma-2-", "2024-07"),
    ("Llama-3.1-", "2024-07"),
    ("Qwen2-", "2024-06"),
    ("Mistral-7B-Instruct-v0.3", "2024-05"),
    ("Llama-3-", "2024-04"),
    ("Phi-3-", "2024-04"),
    ("snowflake-arctic-embed", "2024-04"),
    ("Hermes-2-Theta", "2024-05"),
    ("Hermes-2-Pro", "2024-03"),
    ("gemma-2b-it", "2024-02"),
    ("TinyLlama", "2024-01"),
    ("stablelm-2-zephyr", "2024-01"),
    ("phi-2", "2023-12"),
    ("Mistral-7B-Instruct-v0.2", "2023-12"),
    ("NeuralHermes-2.5", "2023-12"),
    ("WizardMath-7B-V1.1", "2023-12"),
    ("OpenHermes-2.5", "2023-11"),
    ("phi-1_5", "2023-09"),
    ("Llama-2", "2023-07"),
    ("RedPajama", "2023-05"),
]

TYPES = {0: "chat", 1: "embedding", 2: "vision"}


def released(model_id):
    for prefix, month in RELEASED:
        if model_id.lower().startswith(prefix.lower()):
            return month
    return "?"


def collect_models():
    rows = []
    for item in webllm.prebuiltAppConfig.model_list:
        size = getattr(item, "vram_required_MB", None)
        rows.append({
            "name": item.model_id,
            "size": int(size) if size else None,
            "type": TYPES.get(getattr(item, "model_type", None) or 0, "other"),
            "released": released(item.model_id),
        })
    return rows


all_models = collect_models()
print(f"WebLLM knows about {len(all_models)} models.")

In [ ]:
NEWER_THAN = "2025-01"
MAX_SIZE_MB = 4000
MODEL_TYPE = "chat"
ONLY_Q4F16 = True

models = []
for model in all_models:
    if model["type"] != MODEL_TYPE:
        continue
    if ONLY_Q4F16 and "q4f16" not in model["name"]:
        continue
    if not model["size"] or model["size"] > MAX_SIZE_MB:
        continue
    if NEWER_THAN and (model["released"] == "?" or model["released"] < NEWER_THAN):
        continue
    models.append(model)

models.sort(key=lambda model: (model["released"], model["size"]), reverse=True)

print(f"{len(models)} models match your filters (newest first).\n")
print(f"{'#':>3}  {'released':<9} {'size (MB)':>9}  {'saved?':<7} model")
for number, model in enumerate(models, start=1):
    saved = "saved" if await webllm.hasModelInCache(model["name"]) else ""
    print(f"{number:>3}  {model['released']:<9} {model['size']:>9}  {saved:<7} {model['name']}")

### Optional: browse the full model list

This table shows **every** model WebLLM supports. Put a word like `"qwen"`, `"gemma"`, or `"embed"` into `SEARCH` to narrow it down.

In [ ]:
import pandas as pd

SEARCH = ""

table = pd.DataFrame(all_models).rename(columns={"size": "size (MB)"})
table = table.sort_values(["released", "size (MB)"], ascending=[False, True]).reset_index(drop=True)
if SEARCH:
    table = table[table["name"].str.contains(SEARCH, case=False)]

pd.set_option("display.max_rows", None)
print(f"{len(table)} models")
table

## Step 3: Learn how to read a model name

Model names look complicated at first, but they usually contain a few useful clues:

- the **family** name, like Gemma, Llama, Qwen, or SmolLM
- the **size**, like `1B` for about one billion parameters or `500M` for about five hundred million parameters
- a word like **`Instruct`**, which means the model was tuned for question-answering and chat
- the **compression** tag, like **`q4f16`**, which tells you how the model was packed to fit on a smaller machine

The next cell uses a simple set of rules to pull those pieces out. It is not perfect, but it helps you see the pattern.

In [ ]:
import re

EXAMPLE_MODEL = models[0]["name"] if models else "gemma3-1b-it-q4f16_1-MLC"

size_match = re.search(r"(\d+(?:\.\d+)?)([BbMm])", EXAMPLE_MODEL)
compression_match = re.search(r"(q\d+f\d+(?:_\d+)?)", EXAMPLE_MODEL, flags=re.IGNORECASE)

family = EXAMPLE_MODEL
if size_match:
    family = EXAMPLE_MODEL[:size_match.start()].rstrip("-_")

if size_match:
    number, unit = size_match.groups()
    if unit.upper() == "B":
        size_text = f"about {number} billion parameters"
    else:
        size_text = f"about {number} million parameters"
else:
    size_text = "no size pattern found"

if "instruct" in EXAMPLE_MODEL.lower() or "-it-" in EXAMPLE_MODEL.lower():
    tuning_text = "This looks like an instruction-tuned model for chat or question-answering."
else:
    tuning_text = "This name does not clearly say it is instruction-tuned."

print(f"Model name: {EXAMPLE_MODEL}\n")
print(f"Family: {family}")
print(f"Size: {size_text}")
print(f"Compression tag: {compression_match.group(1) if compression_match else 'not found'}")
print(tuning_text)
print("\nTip: q4f16 is a compressed version that is much smaller than the original model, which is why it can fit in a browser.")

## Step 4: Look up the model card on Hugging Face

A **model card** is the model's information page. It usually tells you:

- who made the model
- what data or training recipe was used
- the license
- the intended use
- the limitations, risks, or warnings

For WebLLM models, there are often **two** useful Hugging Face pages:

1. the **original model** from the model maker
2. the **converted model** from `mlc-ai`, which is the version prepared for WebLLM

The next cell prints both link patterns. For some models, the original model name is easy to guess. For others, you may need to search Hugging Face if the names do not match exactly.

In [ ]:
MODEL_CARD_EXAMPLES = {
    "gemma3-1b-it-q4f16_1-MLC": {
        "original": "google/gemma-3-1b-it",
        "converted": "mlc-ai/gemma-3-1b-it-q4f16_1-MLC",
    },
    "Llama-3.2-1B-Instruct-q4f16_1-MLC": {
        "original": "meta-llama/Llama-3.2-1B-Instruct",
        "converted": "mlc-ai/Llama-3.2-1B-Instruct-q4f16_1-MLC",
    },
    "Qwen2.5-0.5B-Instruct-q4f16_1-MLC": {
        "original": "Qwen/Qwen2.5-0.5B-Instruct",
        "converted": "mlc-ai/Qwen2.5-0.5B-Instruct-q4f16_1-MLC",
    },
    "SmolLM2-360M-Instruct-q4f16_1-MLC": {
        "original": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "converted": "mlc-ai/SmolLM2-360M-Instruct-q4f16_1-MLC",
    },
    "snowflake-arctic-embed-m-q0f32-MLC": {
        "original": "Snowflake/snowflake-arctic-embed-m",
        "converted": "mlc-ai/snowflake-arctic-embed-m-q0f32-MLC",
    },
}

MODEL_FOR_CARD = models[0]["name"] if models else "gemma3-1b-it-q4f16_1-MLC"
match = MODEL_CARD_EXAMPLES.get(MODEL_FOR_CARD)

if match:
    original_repo = match["original"]
    converted_repo = match["converted"]
    print("Example found in the helper table.")
else:
    original_repo = "ORG_OR_AUTHOR/MODEL-NAME"
    converted_repo = f"mlc-ai/{MODEL_FOR_CARD}"
    print("No helper entry for this model yet. Fill in ORIGINAL_REPO by hand after searching Hugging Face.")

print(f"\nWebLLM model id: {MODEL_FOR_CARD}")
print(f"Original model card:  https://huggingface.co/{original_repo}")
print(f"Converted model card: https://huggingface.co/{converted_repo}")

## Step 5: Pick a model to download

Type the **number** of a model from the filtered list in Step 2. A small chat model near the top is a good place to start.

In [ ]:
DOWNLOAD_CHOICE = 1

download_name = models[DOWNLOAD_CHOICE - 1]["name"]
download_size = models[DOWNLOAD_CHOICE - 1]["size"]
print(f"You picked: {download_name} (about {download_size} MB)")

## Step 6: Download it

This saves the model in your browser. If it is already saved, nothing is downloaded.

In [ ]:
if await webllm.hasModelInCache(download_name):
    print(f"{download_name} is already saved. Nothing to download.")
else:
    await download_model(download_name)
    print(f"{download_name} is downloaded and saved in your browser.")

## Step 7: Check what you have saved

This lists every model saved in your browser.

In [ ]:
saved_models = []
for model in all_models:
    if await webllm.hasModelInCache(model["name"]):
        saved_models.append((model["name"], model["size"]))

if saved_models:
    print("Models saved in your browser:")
    for number, (name, size) in enumerate(saved_models, start=1):
        print(f"{number:>3}  {name}  ({size or '?'} MB)")
    total = sum(size or 0 for _, size in saved_models)
    print(f"\nTotal: about {total} MB")
else:
    print("No models saved yet. Go back to Steps 5 and 6.")

### Optional: delete a saved model

Put the number of a saved model into `DELETE_CHOICE` and run the next cell to remove it. Leave it at `0` to do nothing.

In [ ]:
DELETE_CHOICE = 0

if DELETE_CHOICE:
    name, _ = saved_models[DELETE_CHOICE - 1]
    await webllm.deleteModelAllInfoInCache(name)
    print(f"Deleted {name}. Run Step 7 again to see the updated list.")
else:
    print("Nothing deleted.")

## Step 8: Other places to browse models

Two other places are useful when you want to explore beyond this notebook:

- WebLLM's model list in `src/config.ts`: <https://github.com/mlc-ai/web-llm/blob/main/src/config.ts>
- The WebLLM Chat demo: <https://chat.webllm.ai/>

The notebook is easier for learning, but those two places are handy when you want to see what is new.